# ML-04 — Search Intelligence Data Contract (assignment-aligned copy)

This is a separate working copy aligned to the ML-04 brief. It uses a mid-panel month for the five honest features and keeps June 2026 sealed.

In [1]:
from huggingface_hub import get_token
import duckdb
import numpy as np

HF_TOKEN = get_token()
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
EXAMPLE_CUTOFF_DATE = "2026-03-31"
EXAMPLE_FEATURE_START = "2026-03-02"
EXAMPLE_TARGET_START = "2026-04-01"
EXAMPLE_TARGET_END = "2026-04-30"
MIN_FEATURE_IMPRESSIONS = 100
DECLINE_RATIO = 0.80

## 1. Unit of analysis + time window

### General contract

1. **What one row means.** In the source, one row should be one measured day for one pseudonymized client and content item. In the modeling frame, one row is one eligible client-content item at one cutoff date `T`.
2. **Table used.** I use only `fact_content_daily_performance`. I do not need a dimension-table join for these five features.
3. **Time window.** The five honest features use the 30 days from `T-29` through `T`. The future proxy uses the next 30 days from `T+1` through `T+30`. The feature and target windows do not overlap.
4. **What I predict and rank.** I predict `is_declining_label`, equal to 1 when impressions in `T+1...T+30` are more than 20% below impressions in `T-29...T`. The model score can rank eligible, visible pages for human review; it does not prove that a refresh will help.
5. **One deliberate exclusion.** I exclude all GA4 fields because availability is limited and unavailable GA4 rows contain zero-filled measurements that do not mean zero engagement.

A candidate must have GSC available on all 30 feature days and at least 100 feature-window impressions. A historical row additionally needs GSC available on all 30 target days so its label can be observed; target completeness is never a model feature.

### One mid-panel cutoff example

For the small feature frame required here, `T = 2026-03-31`. Therefore:

- feature window `T-29...T`: **2026-03-02 through 2026-03-31**;
- target window `T+1...T+30`: **2026-04-01 through 2026-04-30**.

## 2. Fields: feature / label / context / excluded

### Five features

1. `impressions_feature_30d`: GSC impression total from `T-29` through `T`. **Knowable at the decision moment because** it uses only observations on or before `T`.
2. `clicks_feature_30d`: GSC click total from `T-29` through `T`. **Knowable at the decision moment because** it uses only observations on or before `T`.
3. `ctr_feature_30d_pct`: `100 × clicks_feature_30d / impressions_feature_30d`. **Knowable at the decision moment because** both inputs end at `T`.
4. `weighted_position_feature_30d`: `SUM(gsc_sum_position) / SUM(gsc_impressions)` from `T-29` through `T`. **Knowable at the decision moment because** both inputs end at `T`.
5. `impressions_change_late15_vs_early15_pct`: impression change from the first 15 feature days to the last 15. **Knowable at the decision moment because** both halves end by `T`.

### Label / proxy

- `impressions_target_30d` is the observed impression total from `T+1` through `T+30`, used only to construct the label.
- `is_declining_label = 1` when `impressions_target_30d < 0.80 × impressions_feature_30d`; otherwise it is 0.
- `future_impressions_change_pct` is derived from the target window and therefore reveals the label. I add it only for the required leakage demonstration.

### Context

- `client_hash_id` and `content_hash_id`: grouping and joining.
- `feature_gsc_days` and `target_gsc_days`: coverage checks.
- `T`: cutoff construction and later time-aware validation. In this notebook example, `T = 2026-03-31`.

### Excluded

- GA4 availability flags and all GA4/session/AI/scroll fields: excluded from this prototype because of sparse availability and zero-filled unavailable rows.

## 3. Verify it with queries (grain, counts, missing values, windows)

The three numbered cells below are the assignment's three verification queries. All three use the March 2026 partition example.

### Verification query 1 of 3: grain

If `duplicate_key_groups` is zero, March really has at most one source row per report date, client, and content item.

In [2]:
con.sql(f"""
    WITH duplicate_keys AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS rows_at_key
        FROM {MARCH}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_key_groups,
        COALESCE(SUM(rows_at_key - 1), 0) AS extra_duplicate_rows
    FROM duplicate_keys
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_key_groups,extra_duplicate_rows
0,0,0.0


### Verification query 2 of 3: slice count and date span

In [3]:
con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH}
""").df()

,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Verification query 3 of 3: GSC availability

In [4]:
con.sql(f"""
    SELECT
        COUNT(*) AS gsc_available_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id)) AS client_page_pairs_surviving
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_available_rows,client_page_pairs_surviving
0,3611061,176738


### Build the small five-feature frame

This is feature construction. For the example `T = 2026-03-31`, the five feature columns use March 2–31 only. April 1–30 contributes only the observable future label.

In [5]:
feature_frame = con.sql(f"""
    WITH feature_page AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_feature_30d,
            SUM(gsc_clicks) AS clicks_feature_30d,
            SUM(gsc_sum_position) AS position_sum_feature_30d,
            SUM(gsc_impressions) FILTER (
                WHERE report_date <= DATE '{EXAMPLE_CUTOFF_DATE}' - INTERVAL 15 DAY
            ) AS impressions_early15,
            SUM(gsc_impressions) FILTER (
                WHERE report_date > DATE '{EXAMPLE_CUTOFF_DATE}' - INTERVAL 15 DAY
            ) AS impressions_late15
        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '{EXAMPLE_FEATURE_START}'
                              AND DATE '{EXAMPLE_CUTOFF_DATE}'
        GROUP BY 1, 2
        HAVING COUNT(DISTINCT report_date) = 30
    ),
    target_page AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_target_30d
        FROM {APRIL}
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '{EXAMPLE_TARGET_START}'
                              AND DATE '{EXAMPLE_TARGET_END}'
        GROUP BY 1, 2
        HAVING COUNT(DISTINCT report_date) = 30
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        DATE '{EXAMPLE_CUTOFF_DATE}' AS cutoff_date,
        f.impressions_feature_30d,
        f.clicks_feature_30d,
        100.0 * f.clicks_feature_30d
            / NULLIF(f.impressions_feature_30d, 0) AS ctr_feature_30d_pct,
        f.position_sum_feature_30d
            / NULLIF(f.impressions_feature_30d, 0) AS weighted_position_feature_30d,
        100.0 * (f.impressions_late15 - f.impressions_early15)
            / NULLIF(f.impressions_early15, 0)
            AS impressions_change_late15_vs_early15_pct,
        t.impressions_target_30d,
        CASE
            WHEN t.impressions_target_30d < {DECLINE_RATIO} * f.impressions_feature_30d THEN 1
            ELSE 0
        END AS is_declining_label
    FROM feature_page f
    JOIN target_page t USING (client_hash_id, content_hash_id)
    WHERE f.impressions_feature_30d >= {MIN_FEATURE_IMPRESSIONS}
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

FEATURES = [
    "impressions_feature_30d",
    "clicks_feature_30d",
    "ctr_feature_30d_pct",
    "weighted_position_feature_30d",
    "impressions_change_late15_vs_early15_pct",
]

print(f"Example cutoff: {EXAMPLE_CUTOFF_DATE}")
print(f"Feature window: {EXAMPLE_FEATURE_START} through {EXAMPLE_CUTOFF_DATE}")
print(f"Target window: {EXAMPLE_TARGET_START} through {EXAMPLE_TARGET_END}")
print(f"Feature frame: {len(feature_frame):,} eligible client-page rows")
print(f"Observed decline-proxy rate: {feature_frame['is_declining_label'].mean():.1%}")
print("Missing values in the five honest features:")
display(feature_frame[FEATURES].isna().sum().rename("missing_rows").to_frame())
display(feature_frame[FEATURES + ["is_declining_label"]].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Example cutoff: 2026-03-31
Feature window: 2026-03-02 through 2026-03-31
Target window: 2026-04-01 through 2026-04-30
Feature frame: 50,641 eligible client-page rows
Observed decline-proxy rate: 47.3%
Missing values in the five honest features:


,missing_rows
impressions_feature_30d,0
clicks_feature_30d,0
ctr_feature_30d_pct,0
weighted_position_feature_30d,0
impressions_change_late15_vs_early15_pct,0


,impressions_feature_30d,clicks_feature_30d,ctr_feature_30d_pct,weighted_position_feature_30d,impressions_change_late15_vs_early15_pct,is_declining_label
0,325.0,2.0,0.615385,14.529231,64.227642,0
1,446.0,0.0,0.000000,14.273543,-39.568345,0
2,746.0,5.0,0.670241,12.805630,-41.276596,0
3,3214.0,19.0,0.591164,7.054449,40.749064,0
4,2021.0,1.0,0.049480,53.063335,-50.738552,1


### Availability 

- `impressions_feature_30d`: available once GSC reporting through `T` is complete.
- `clicks_feature_30d`: available once GSC reporting through `T` is complete.
- `ctr_feature_30d_pct`: available then because it is calculated only from clicks and impressions through `T`.
- `weighted_position_feature_30d`: available then because it is calculated only from position sums and impressions through `T`.
- `impressions_change_late15_vs_early15_pct`: available then because both 15-day totals end by `T`.

### The deliberate leakage trap

First I fit the quick depth-3 tree using only the five honest `T-29...T` features. Then I add `future_impressions_change_pct`, which uses `T+1...T+30` and is the answer in disguise. I compare both models on the same split, delete the leaked column, and retain the honest feature list.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

LABEL = "is_declining_label"
model_frame = feature_frame.dropna(subset=FEATURES + [LABEL]).copy()

LEAKED_FEATURE = "future_impressions_change_pct"
model_frame[LEAKED_FEATURE] = (
    100.0 * (model_frame["impressions_target_30d"] - model_frame["impressions_feature_30d"])
    / model_frame["impressions_feature_30d"]
)

train_rows, test_rows = train_test_split(
    model_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=model_frame[LABEL],
)

def score(columns):
    tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
    tree.fit(model_frame.loc[train_rows, columns], model_frame.loc[train_rows, LABEL])
    probabilities = tree.predict_proba(model_frame.loc[test_rows, columns])[:, 1]
    y_test = model_frame.loc[test_rows, LABEL]
    top_50 = np.argsort(probabilities)[-50:]
    return y_test.iloc[top_50].mean()

honest_p50 = score(FEATURES)
leaky_p50 = score(FEATURES + [LEAKED_FEATURE])

print(f"Honest tree Precision@50: {honest_p50:.3f}")
print(f"Leaky tree Precision@50: {leaky_p50:.3f}")

del model_frame[LEAKED_FEATURE]
print(f"Removed leaked column: {LEAKED_FEATURE}")
print("Final honest feature list:", FEATURES)

Honest tree Precision@50: 0.860
Leaky tree Precision@50: 1.000
Removed leaked column: future_impressions_change_pct
Final honest feature list: ['impressions_feature_30d', 'clicks_feature_30d', 'ctr_feature_30d_pct', 'weighted_position_feature_30d', 'impressions_change_late15_vs_early15_pct']


`future_impressions_change_pct` makes the score look exceptional because the label is defined from that same future change. It would not be known when pages are ranked, so the final model excludes it and keeps the honest score.

## 4. Data limits

This single March-to-April example cannot distinguish lasting page decline from seasonality or short-term noise. It also cannot show that refreshing a page would cause recovery; the result is only a measured proxy for review prioritization.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.